<img style="float: left; margin: 30px 15px 15px 15px;" src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="400" height="600" /> 
    
    
## <font color='navy'> Teoría Pos-Moderna de Portafolios: Práctica
    
### <font color='navy'> Portafolios de Inversión

    Mtro. Sean Nicolás González Vázquez

---

### <font color='navy'> 0.- Introducción
    
En la sesión anterior, exploramos varios métodos de la **Teoría Posmoderna de Portafolios (TPMP)**, que se enfoca en la optimización de portafolios separando claramente el riesgo en dos componentes: el downside y el upside risk. 
    
A grandes rasgos, esta teoría amplía la clásica Teoría Moderna de Portafolios, al poner énfasis en los riesgos que afectan negativamente las inversiones, y aquellos que representan oportunidades de ganancia.

La principal contribución de este enfoque radica en su capacidad para descomponer el riesgo en dos categorías clave:

+ **Downside risk**: Riesgo de incurrir en pérdidas, es decir, cuando los rendimientos caen por debajo de un umbral establecido, como podría ser un benchmark o simplemente rendimientos negativos.   
    
    
+ **Upside risk**: Riesgo (potencial) de obtener ganancias.   
    
    
+ Mientras que el downside implica posibles pérdidas para el portafolio, el upside representa las oportunidades de obtener rendimientos positivos.

Dentro de esta estructura, profundizamos en **dos métodos de Asset Allocation** basados en esta separación de riesgos: el método de la **Mínima Semivarianza** y el método de **Máximo Omega**. 
    
Ambos enfoques proporcionan alternativas interesantes a los modelos tradicionales propuestos por Markowitz y Sharpe.   
    
El primero, la Mínima Semivarianza, es una extensión del clásico método de Mínima Varianza, ya que se centra exclusivamente en minimizar el riesgo a la baja (downside risk), ignorando las fluctuaciones al alza. El segundo, el Máximo Omega, es una alternativa al Ratio de Sharpe, con la diferencia clave de que se enfoca en maximizar la relación entre las ganancias potenciales (upside) y las pérdidas posibles (downside), proporcionando una evaluación más matizada del rendimiento ajustado al riesgo.  

> Con este contexto, implementaremos ambos métodos, utilizando optimización montecarlo.

---

### <font color='navy'> 1.- Descarga de Datos y Obtención de Métricas

En este ejercicio, vamos a construir un portafolio compuesto por siete activos financieros previamente seleccionados:

+ PG: Procter & Gamble
+ COST: Costco Wholesale
+ KO: Coca-Cola
+ WMT: Walmart
+ CLX: Clorox
+ K: Kellogg's
+ KHC: Kraft Heinz

Como puedes observar, todos estos activos pertenecen a compañías que producen bienes de consumo básico, tales como productos de limpieza, alimentos y bebidas, entre otros. 


Estos activos forman parte de un sector específico conocido como Consumer Staples. Este sector es considerado anticíclico por naturaleza. Por ello, se le clasifica como un sector defensivo, ideal para inversiónistas que buscan estabilidad y son aversos al riesgo.

> **Consumer Staples:** Hace referencia a bienes de consumo necesarios para la vida diaria, como alimentos, bebidas, productos de higiene personal y del hogar. Estos productos suelen ser demandados de forma constante, independientemente del ciclo económico.  

> **Sector Defensivo:** Se refiere a aquellos sectores de la economía cuyos activos tienden a ser menos volátiles y más resilientes en tiempos de crisis o desaceleraciones económicas. Las empresas en estos sectores producen bienes y servicios esenciales, lo que les permite mantener una demanda relativamente estable incluso en condiciones adversas del mercado.


Como gestor del portafolio (portfolio manager), tu objetivo es encontrar las ponderaciones óptimas que minimicen la semivarianza del portafolio y maximicen el ratio de Omega.

In [1]:
# Importación de Librerías
import numpy as np
import pandas as pd
import yfinance as yf

In [2]:
# Descarga de Datos
prices=yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'], 
                    start='2020-01-01', end='2026-10-07')['Close']

/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_89605/3213264533.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices=yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
[*********************100%***********************]  6 of 6 completed


In [3]:
# Calculo de rendimientos
rets=prices.pct_change().dropna()


In [4]:
# Calcular rendimiento promedio
mu=rets.mean()

In [5]:
# Calcular matriz de correlación
corr=rets.corr()
corr


Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,1.000000,0.319714,0.378644,0.340597,0.506519,0.305476
COST,0.319714,1.000000,0.302888,0.425470,0.471215,0.613987
KHC,0.378644,0.302888,1.000000,0.542350,0.524965,0.299264
KO,0.340597,0.425470,0.542350,1.000000,0.634627,0.370660
PG,0.506519,0.471215,0.524965,0.634627,1.000000,0.464734
WMT,0.305476,0.613987,0.299264,0.370660,0.464734,1.000000


---

### <font color='navy'> 2.- Portafolio Eficiente en Mínima Semivarianza  
    
  
### <font color='navy'>  **Mínima Semivarianza**

Método de la optimización de portafolios que busca minimizar el potencial riesgo de pérdida (*downside risk*) de un portafolio de inversión. ´


**Separar los rendimientos por debajo de 0 (pérdidas), las ganancias convertirlas en cero.**

$$R_{below, i} = min(r_i, 0)$$


In [6]:
below_zero=(rets[rets<0]).fillna(0)
below_zero


Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2020-01-03,0.000000,0.000000,-0.011705,-0.005456,-0.006726,-0.008828
2020-01-06,0.000000,0.000000,0.000000,-0.000366,0.000000,-0.002036
2020-01-07,-0.012063,-0.001577,-0.017566,-0.007682,-0.006192,-0.009265
2020-01-08,0.000000,0.000000,-0.005527,0.000000,0.000000,-0.003432
2020-01-09,0.000000,0.000000,-0.000327,0.000000,0.000000,0.000000
...,...,...,...,...,...,...
2026-09-30,-0.007230,-0.015412,-0.031130,-0.008752,-0.020496,-0.026966
2026-10-01,-0.004444,0.000000,-0.011444,0.000000,-0.009155,0.000000
2026-10-02,-0.002976,0.000000,-0.012021,-0.005226,0.000000,0.000000


**Calcular downside risk para cada activo.**

$$\sigma_{d, i} = \sigma(R_{below, i})$$

In [7]:
downside=np.array(below_zero.std())
downside

array([0.01102385, 0.00896306, 0.01059142, 0.00797065, 0.00809466,
       0.00909594])

**Calcular matriz de Semivarianza.**

Antes de encontrar la matriz de semivarianza, simplifiquemos el cálculo......

Recordando, la varianza de dos activos financieros si tenemos la correlación entre ambos y sus desviaciones estandár individuales esta dada por $\sigma^2_{i, j} = \rho_{i, j} \sigma_i \sigma_j$. Para calcular la semi-varianza entre dos activos financieros utilizamos la misma fórmula, simplemente cambiando la desviación estandár por la semi-desviación (el *downside risk*), es decir, la semivarianza para dos activos financieros esta dada por $\sigma^2_{d(i, j)} = \rho_{i, j} \sigma_{(d, i)} \sigma_{(d, j)}$.

Esto debe hacerse iterativamente hasta construir la matriz de semivarianza, como vimos en la clase pasada. 

Para simplificar el ejercicio, podemos tomar el vector de $nx1$ de los downside risk individuales:

$$\sigma_d = \begin{bmatrix}
\sigma_{d, 1} \\
\sigma_{d, 2} \\
\vdots \\
\sigma_{d, n}
\end{bmatrix}$$

Y multiplicarla matricialmente por su transpuesta:

$$\sigma_d \sigma_d^T = 
\begin{bmatrix}
\sigma_{d, 1} \\
\sigma_{d, 2} \\
\vdots \\
\sigma_{d, n}
\end{bmatrix}
\begin{bmatrix}
\sigma_{d, 1} & \sigma_{d, 2} & \cdots & \sigma_{d, n}
\end{bmatrix}
=
\begin{bmatrix}
\sigma_{d, 1}^2 & \sigma_{d, 1} \sigma_{d, 2} & \cdots & \sigma_{d, 1} \sigma_{d, n} \\
\sigma_{d, 2} \sigma_{d, 1} & \sigma_{d, 2}^2 & \cdots & \sigma_{d, 2} \sigma_{d, n} \\
\vdots & \vdots & \ddots & \vdots \\
\sigma_{d, n} \sigma_{d, 1} & \sigma_{d, n} \sigma_{d, 2} & \cdots & \sigma_{d, n}^2
\end{bmatrix}$$

Notesé que esto equivale a multiplicar el elemento $i$ con el $j$ de la matriz de semivarianza, donde la diagonal son los downside al cuadrado (puedes utilizar la función de `np.multiply`). Finalmente realizamos el producto Hadamard de esta matriz con la matriz de correlación y con esto obtenemos la matriz de semivarianza.

$$S = \begin{bmatrix}
1 & \rho_{12} & \cdots & \rho_{1n} \\
\rho_{21} & 1 & \cdots & \rho_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
\rho_{n1} & \rho_{n2} & \cdots & 1
\end{bmatrix} \circ \begin{bmatrix}
\sigma_{d, 1}^2 & \sigma_{d, 1} \sigma_{d, 2} & \cdots & \sigma_{d, 1} \sigma_{d, n} \\
\sigma_{d, 2} \sigma_{d, 1} & \sigma_{d, 2}^2 & \cdots & \sigma_{d, 2} \sigma_{d, n} \\
\vdots & \vdots & \ddots & \vdots \\
\sigma_{d, n} \sigma_{d, 1} & \sigma_{d, n} \sigma_{d, 2} & \cdots & \sigma_{d, n}^2
\end{bmatrix}$$

El producto [Hadamard](https://en.wikipedia.org/wiki/Hadamard_product_(matrices)#:~:text=In%20mathematics,%20the%20Hadamard%20product%20also%20known%20as%20the) es una multiplicación elemento a elemento de cada matriz según su posición específica (para realizarla en python, puedes utilizar simplimente `*`). Al realizar esta operación obtenemos la matriz de semivarianza para el portafolio:

$$S=
\begin{bmatrix}
\sigma_{d, 1}^2 & \rho_{12} \sigma_{d, 1} \sigma_{d, 2} & \cdots & \rho_{1n} \sigma_{d, 1} \sigma_{d, n} \\
\rho_{21} \sigma_{d, 2} \sigma_{d, 1} & \sigma_{d, 2}^2 & \cdots & \rho_{2n} \sigma_{d, 2} \sigma_{d, n} \\
\vdots & \vdots & \ddots & \vdots \\
\rho_{n1} \sigma_{d, n} \sigma_{d, 1} & \rho_{n2} \sigma_{d, n} \sigma_{d, 2} & \cdots & \sigma_{d, n}^2
\end{bmatrix}$$


In [8]:
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix=(downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*corr
semivar_matrix

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,0.000122,0.000032,0.000044,0.000030,0.000045,0.000031
COST,0.000032,0.000080,0.000029,0.000030,0.000034,0.000050
KHC,0.000044,0.000029,0.000112,0.000046,0.000045,0.000029
KO,0.000030,0.000030,0.000046,0.000064,0.000041,0.000027
PG,0.000045,0.000034,0.000045,0.000041,0.000066,0.000034
WMT,0.000031,0.000050,0.000029,0.000027,0.000034,0.000083


**Calcular semivarianza del portafolio.**

Con la matriz de semivarianza, podemos obtener la semivarianza de un portafolio de inversión con la fórmula:

$$s = w^T S w$$

In [9]:
semivar_port=lambda w: w.T @ semivar_matrix @ w
n_sim=10000
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [ ]:
temp_var=[]

    temp_w=np.random.dirichlet(np.ones(n_assets),size=n_sim)
    temp_random=[lambda w: w.T @ semivar_matrix @ w]
    temp_var.append(min(temp_random))

[<function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __main__.<lambda>(w)>,
 <function __m

**Minimizar semivarianza cambiando ponderaciones.**

$$min_w \hspace{0.5cm} \sigma_d^2 = w^T S w$$
    
$$s.a. \hspace{0.5cm} \sum_{i=1}^n w_i = 1$$
 
$$\hspace{0.8cm} w_i > 0 $$



In [ ]:
# Simular portafolios con montecarlo




In [ ]:
# Obtener la mínima semivarianza


In [ ]:
# Obtener los pesos del portafolio con mínima semivarianza




---

### <font color='navy'> 3.- Portafolio Eficiente en Máximo Omega
    
    
Portafolio que maximiza el ratio omega de un portafolio de inversión, el ratio Omega mide la relación entre el upside risk (potencial de ganancias) y el downside risk (potencial de pérdidas) de un activo financiero.

**Separar los rendimientos en dos, aquellos por debajo de cero y los que están por encima de cero.**

$$R_{below, i} = min(r_i, 0)$$

$$R_{beneath, i} = max(r_i, 0)$$

**Calcular downside y upside risk para cada activo.**

$$\sigma_{d, i} = \sigma(R_{below, i})$$

$$\sigma_{u, i} = \sigma(R_{beneath, i})$$

**Calcular Omega individual para cada activo.**

$$\Omega_i = \frac{\sigma_{u, i}}{\sigma_{d, i}}$$


**Calcular Omega del portafolio y maximizarla cambiando ponderaciones.**

$$max_w \hspace{0.5cm} \Omega_p = \sum_{i=1}^{n} \Omega_i * w_i$$
    
$$s.a. \hspace{0.5cm} \sum_{i=1}^n w_i = 1$$
 
$$\hspace{0.8cm} w_i > 0 $$

In [ ]:
# Simular portafolios con montecarlo





In [ ]:
# Obtener el maximo omega


In [ ]:
# Obtener los pesos del portafolio con maximo omega




---

### <font color='navy'> 4.- Target Minimum Semivariance Portfolio
    

Ahora, supón que deseas optimizar las ponderaciones del portafolio considerando aquellos activos que tienen mayor probabilidad de superar a tu *benchmark*. Para ello, emplearás una estrategia basada en la *target semivariance*.

En este caso, asume que tu *benchmark* es el ETF `KXI`, que sigue el sector de bienes de consumo global. 
    
El objetivo es encontrar las ponderaciones óptimas que minimicen el riesgo de no superar el rendimiento de `KXI`. 
    
    
Recuerda que este enfoque ajustado al *benchmark* permite identificar aquellos activos que tienen un mejor desempeño relativo y, por lo tanto, una mayor probabilidad de generar *alpha*. Al minimizar la semivarianza relativa al *benchmark*, estarás optimizando el portafolio para que minimice el riesgo de rendimientos inferiores al de `KXI`.

Recuerda que los pasos son reiterativos y siguen la misma estructura que el método de mínima semivarianza, solo que ahora restarás los rendimientos al rendimiento del *benchmark* a los de tus activos.

In [ ]:
# Descarga de Precios del Benchmark


In [ ]:
# Obtener rendimientos del benchmark




In [ ]:
# Sustraer rendimiento del benchmark de rendimientos individuales




In [ ]:
# Separar los rendimientos por debajo de cero




In [ ]:
# Calcular target downside risk




In [ ]:
# Calcular la matriz de target semivariance




In [ ]:
# Simular portafolios con montecarlo




In [ ]:
# Obtener la mínima target semivariance


In [ ]:
# Obtener los pesos del portafolio con mínima semivarianza target




    ¡Excelente! Ahora comparemos las ponderaciones obtenidas en el método de semivarianza normal con target semivariance, responde: ¿Hay diferencias? ¿Como las interpretas?

---

### <font color='navy'> 5.- Reflexión

> **El método de la semivarianza y el ratio omega** son dos alternativas interesantes para la optimización de portafolios de inversión, ya que **replantean el concepto tradicional del riesgo financiero**.

> **¿Cuál es mejor?** No hay una respuesta definitiva; **la elección** del método "adecuado" **depende de varios factores**, como el perfil del inversionista, el comportamiento histórico de los activos seleccionados (que puede hacer que un método funcione mejor que otro) y las restricciones u objetivos específicos del gestor.

> Es importante resaltar nuevamente que estos **métodos solo son tan efectivos como lo sean los activos incluidos en el portafolio**.

> Hasta ahora, no hemos comparado las distintas estrategias de asignación de activos, sino que hemos enfocado en cómo determinar los pesos óptimos. **En las próximas sesiones, nos centraremos en la comparación y selección de estrategias de inversión**. Esto te permitirá, con un conjunto de activos que conforman un portafolio, evaluar sus características y elegir la mejor estrategia basándote en las métricas de los múltiples portafolios óptimos.

---

### <font color='navy'> Tarea 5

En equipos de dos integrantes, respondan los siguientes puntos y entreguen un archivo `html` con sus resultados. No olviden incluir los nombres de los integrantes en el archivo:

**Punto a)** Implementen tres funciones que optimicen portafolios utilizando: mínima semivarianza, ratio omega y semivarianza objetivo, empleando la función `scipy.optimize.minimize`.

**Punto b)** Construyan un portafolio compuesto por 5 activos financieros y optimícenlo con los tres métodos. Utilicen las funciones desarrolladas en el punto a) para realizar la optimización. Recuerden que, para el portafolio con semivarianza objetivo, deberán seleccionar un benchmark adecuado y consistente con los activos elegidos. Como respuesta a este punto, se esperan las ponderaciones eficientes y el valor óptimo de la función objetivo para cada método de Asset Allocation.

**Punto c)** Escriban una breve conclusión sobre las ventajas y desventajas que observan entre los métodos TMP y TPMP.

**Extra:** El equipo que logre la menor semivarianza global en su portafolio optimizado por mínima semivarianza recibirá 10 puntos extra para el segundo examen parcial. La selección de activos con una buena relación entre upside y downside será clave para obtener estos puntos.

